# all-reduce-grad-sync — worked example 3: Mean-reduce grads across three ranks

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-grad-sync`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The SUM-then-divide pattern generalizes to any `world_size`. With three ranks the all_reduce sums all three per-rank grads and the divisor becomes 3. The result on every rank is the arithmetic mean of the three local gradients.

## Worked solution

This drill shows the world_size divisor is not hard-coded to 2 - it must equal the number of ranks.

1. Create three per-rank grads: `[2, 0]`, `[0, 6]`, `[4, 0]`. Their element-wise SUM is `[6, 6]`.
2. The mock `all_reduce` writes that SUM back into all three buffers, matching what a real 3-process group would do.
3. We divide by `world_size = 3`, giving the mean `[2, 2]` on every rank.
4. We verify against an independent `numpy` mean of the original three grads. They match, confirming SUM/world_size is exactly the mean.

The lesson: always divide by the *actual* world_size; using the wrong divisor scales the learning signal incorrectly and breaks the equivalence to single-GPU training.

In [ ]:
import numpy as np
import torch as t

t.manual_seed(0)
world_size = 3
original = [t.tensor([2.0, 0.0]), t.tensor([0.0, 6.0]), t.tensor([4.0, 0.0])]
grads = [g.clone() for g in original]

total = sum(g.clone() for g in grads)   # all_reduce SUM
for g in grads:
    g.copy_(total)
    g /= world_size                      # -> mean

expected = np.mean([g.numpy() for g in original], axis=0)
print('synced grad (all ranks):', grads[0].tolist())
print('numpy mean check       :', expected.tolist())